# 🌋 Seismic Precursors of LLM Degradation

**Applying fractal/seismic methodology to AI telemetry**

This notebook demonstrates how Long-Range Dependence (LRD) and Hurst exponent — the same metrics used to predict earthquakes — can detect LLM degradation **before** visible repetition/looping.

## Key Insight

> When the model starts looping, repetition rate rises sharply — but **Hurst exponent and entropy often shift earlier**, acting as "seismic precursors" before the visible collapse.

### Related Publications (Zenodo)

| Record | DOI |
|--------|-----|
| QO3/FIO v2.2 | [10.5281/zenodo.18145167](https://doi.org/10.5281/zenodo.18145167) |
| LRD Analysis | [10.5281/zenodo.18110450](https://doi.org/10.5281/zenodo.18110450) |
| Fractal Operator Ψ | [10.5281/zenodo.18102168](https://doi.org/10.5281/zenodo.18102168) |

**Community:** [zenodo.org/communities/lrd-time-series](https://zenodo.org/communities/lrd-time-series/records)

---

**Author:** Igor Chechelnitsky • [ORCID: 0009-0007-4607-1946](https://orcid.org/0009-0007-4607-1946)

In [ ]:
# Install dependencies
!pip -q install transformers accelerate matplotlib torch numpy scipy

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import List, Optional, Tuple
from dataclasses import dataclass, field

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Define Metrics

We implement four key metrics:
1. **Token Entropy** — uncertainty in next-token prediction
2. **Repetition Rate** — fraction of repeated tokens (late signal)
3. **Hurst Exponent** — fractal memory / long-range dependence (early signal)
4. **SID** — Seismic Information Deficit (cumulative entropy loss)

In [ ]:
def token_entropy_from_logits(logits: torch.Tensor) -> float:
    """Shannon entropy for next-token distribution."""
    probs = torch.softmax(logits, dim=-1)
    ent = -(probs * torch.log(probs + 1e-12)).sum().item()
    return float(ent)


def repetition_rate(token_ids: List[int], window: int = 200) -> float:
    """Fraction of tokens in last window that appeared earlier."""
    if len(token_ids) < 20:
        return 0.0
    tail = token_ids[-min(window, len(token_ids)):]
    seen_before = set(token_ids[:-len(tail)])
    if not seen_before:
        return 0.0
    return sum(1 for t in tail if t in seen_before) / len(tail)


def hurst_exponent(ts: np.ndarray, max_lag: Optional[int] = None) -> float:
    """
    Hurst exponent via R/S analysis.
    
    H ≈ 0.5: Random walk (healthy)
    H > 0.65: Persistent / trending (warning)
    H > 0.80: Near-deterministic (collapse)
    """
    ts = np.asarray(ts, dtype=np.float64)
    if len(ts) < 20:
        return np.nan
    
    if max_lag is None:
        max_lag = len(ts) // 4
    max_lag = max(max_lag, 10)
    lags = np.arange(10, min(max_lag + 1, len(ts) // 2))
    
    if len(lags) < 3:
        return np.nan
    
    rs_values, valid_lags = [], []
    
    for lag in lags:
        n_blocks = len(ts) // lag
        if n_blocks < 2:
            continue
        
        rs_block = []
        for i in range(n_blocks):
            block = ts[i * lag:(i + 1) * lag]
            if len(block) < 2:
                continue
            mean_block = np.mean(block)
            cumdev = np.cumsum(block - mean_block)
            R = np.max(cumdev) - np.min(cumdev)
            S = np.std(block, ddof=1)
            if S > 1e-10:
                rs_block.append(R / S)
        
        if rs_block:
            rs_values.append(np.mean(rs_block))
            valid_lags.append(lag)
    
    if len(rs_values) < 3:
        return np.nan
    
    slope, _, _, _, _ = stats.linregress(np.log(valid_lags), np.log(rs_values))
    return float(np.clip(slope, 0, 1))


def nucleus_sample_next_token(logits: torch.Tensor, temperature: float = 1.0, top_p: float = 0.95) -> int:
    """Nucleus (top-p) sampling."""
    logits = logits / max(temperature, 1e-6)
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    sorted_probs = torch.softmax(sorted_logits, dim=-1)
    cum = torch.cumsum(sorted_probs, dim=-1)
    cutoff = cum > top_p
    cutoff[0] = False
    sorted_logits[cutoff] = -1e10
    filtered_probs = torch.softmax(sorted_logits, dim=-1)
    pick = torch.multinomial(filtered_probs, 1).item()
    return int(sorted_idx[pick].item())


print("✅ Metrics defined")

## 2. Load Model

In [ ]:
MODEL = 'distilgpt2'  # Small model for demo; try 'gpt2-medium' or larger
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Loading {MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
model.eval()
print(f"✅ Model loaded on {device}")

## 3. Generate with Telemetry

We use a prompt designed to encourage repetition/looping.

In [ ]:
# Configuration
prompt = (
    "Repeat the explanation with more detail and examples, "
    "do not stop, keep expanding.\n\n"
    "Explain time series, regime shifts, and why early warning "
    "signals matter. Describe how complex systems can suddenly "
    "transition from one state to another.\n\n"
)

max_new_tokens = 2500
sliding_window = 800
rep_threshold = 0.55
hurst_warning = 0.65
temperature = 1.0
top_p = 0.95

print(f"Generating {max_new_tokens} tokens...")
print("-" * 50)

In [ ]:
# Generation loop with telemetry
full_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)[0]
ent_series, rep_series = [], []
hurst_positions, hurst_series = [], []

with torch.no_grad():
    for step in range(max_new_tokens):
        ctx = full_ids[-sliding_window:] if len(full_ids) > sliding_window else full_ids
        out = model(ctx.unsqueeze(0))
        logits = out.logits[0, -1, :]
        
        # Record entropy
        ent_series.append(token_entropy_from_logits(logits))
        
        # Sample next token
        next_id = nucleus_sample_next_token(logits, temperature=temperature, top_p=top_p)
        full_ids = torch.cat([full_ids, torch.tensor([next_id], device=device)])
        
        # Record repetition
        rep_series.append(repetition_rate(full_ids.tolist(), window=200))
        
        # Calculate Hurst every 100 steps
        if step > 200 and step % 100 == 0:
            h = hurst_exponent(np.array(ent_series[-200:]))
            if not np.isnan(h):
                hurst_positions.append(step)
                hurst_series.append(h)
                if h > hurst_warning:
                    print(f"⚠️  Step {step}: Hurst = {h:.3f} (warning!)")
        
        # Progress
        if step % 500 == 0 and step > 0:
            print(f"Step {step}/{max_new_tokens}")

text = tokenizer.decode(full_ids, skip_special_tokens=True)
print("-" * 50)
print(f"✅ Generation complete: {len(ent_series)} tokens")

## 4. Analyze Results

In [ ]:
# Find collapse and warning points
first_collapse = next((i for i, v in enumerate(rep_series) if v > rep_threshold), None)
first_warning = next((i for i, h in zip(hurst_positions, hurst_series) if h > hurst_warning), None)

print("=" * 60)
print("ANALYSIS RESULTS")
print("=" * 60)

if first_collapse is not None:
    print(f"\n⚠️  Repetition collapse at step: {first_collapse}")
    start = max(0, first_collapse - 100)
    mean_ent = np.mean(ent_series[start:first_collapse])
    print(f"   Mean entropy before: {mean_ent:.4f}")
else:
    print("\n✅ No repetition collapse detected")

if first_warning is not None:
    print(f"\n🔔 Early warning (Hurst > {hurst_warning}) at step: {first_warning}")
    if first_collapse is not None and first_warning < first_collapse:
        lead_time = first_collapse - first_warning
        print(f"   ⏱️  Lead time: {lead_time} tokens before collapse!")
else:
    print(f"\n✅ No Hurst warning detected")

if hurst_series:
    print(f"\nFinal Hurst exponent: {hurst_series[-1]:.4f}")

print("=" * 60)

In [ ]:
# Show last part of text
print("\n=== LAST 1000 CHARS (check for looping) ===")
print(text[-1000:])

## 5. Visualizations

In [ ]:
# Combined dashboard
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Entropy
ax = axes[0]
ax.plot(ent_series, color='#2E86AB', linewidth=0.8, alpha=0.7)
window = 50
rolling = np.convolve(ent_series, np.ones(window)/window, mode='valid')
ax.plot(range(window-1, len(ent_series)), rolling, color='#E94F37', linewidth=2, label='Rolling mean')
ax.set_ylabel('Entropy')
ax.set_title('Token Entropy (lower = more confident = potential precursor)')
ax.legend()
ax.grid(True, alpha=0.3)

# Repetition
ax = axes[1]
ax.plot(rep_series, color='#2E86AB', linewidth=0.8)
ax.axhline(rep_threshold, color='#E94F37', linestyle='--', linewidth=2, label=f'Threshold ({rep_threshold})')
ax.set_ylabel('Repetition Rate')
ax.set_ylim(0, 1)
ax.set_title('Repetition Rate (visible collapse)')
ax.legend()
ax.grid(True, alpha=0.3)

# Hurst
ax = axes[2]
if hurst_positions and hurst_series:
    ax.plot(hurst_positions, hurst_series, 'o-', color='#2E86AB', linewidth=2, markersize=6)
    ax.axhline(hurst_warning, color='#F6AE2D', linestyle='--', linewidth=2, label=f'Warning ({hurst_warning})')
    ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='Random walk')
ax.set_ylabel('Hurst (H)')
ax.set_ylim(0.3, 1.0)
ax.set_xlabel('Generation Step')
ax.set_title('Hurst Exponent (fractal memory — EARLY WARNING)')
ax.legend()
ax.grid(True, alpha=0.3)

# Add collapse/warning markers
for ax in axes:
    if first_warning is not None:
        ax.axvline(first_warning, color='#F6AE2D', linestyle='--', linewidth=2, alpha=0.7)
    if first_collapse is not None:
        ax.axvline(first_collapse, color='#E94F37', linestyle='-', linewidth=2, alpha=0.7)

plt.tight_layout()
plt.suptitle('🌋 Seismic Precursors of LLM Degradation', fontsize=14, fontweight='bold', y=1.02)
plt.show()

## 6. Interpretation

### What the plots show:

1. **Entropy (top)**: Falls as the model becomes "too confident" — often drops before visible looping

2. **Repetition (middle)**: Direct measure of looping — but this is a **late** signal

3. **Hurst Exponent (bottom)**: When H rises above 0.65, the entropy time series shows **long-range dependence** — the model is entering a "persistent" regime that often precedes collapse

### The "Seismic" Analogy:

| Earthquake Science | LLM Degradation |
|-------------------|------------------|
| b-value changes | Entropy distribution shifts |
| Foreshocks | Hurst exponent increase |
| Mainshock | Visible repetition/looping |
| LRD in seismic catalog | LRD in entropy time series |

---

**Cite this work:**
```
Chechelnitsky, I. (2026). Seismic Precursors of LLM Degradation: LRD/QO3/FIO Bridge.
Zenodo. https://doi.org/10.5281/zenodo.18145167
```